# Traffic Flow Prediction using Stacked LSTM

This cleaned portfolio notebook evaluates the supplied pretrained Stacked LSTM on the deterministic hourly traffic dataset. It uses the same modular code as the Streamlit application.

> **Responsible use:** Educational portfolio demonstration only. Do not use these forecasts as the sole basis for traffic control, public safety, emergency response, or transportation policy.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.forecasting_pipeline import TrafficForecastingPipeline
from src.synthetic_data import generate_traffic_data

## Generate the deterministic dataset

In [ ]:
traffic = generate_traffic_data()
traffic.head(), traffic.shape

## Chronological test period

The original notebook used a 70% / 15% / 15% chronological split. No random shuffle is used.

In [ ]:
test_start = int(len(traffic) * 0.85)
test = traffic.iloc[test_start:].copy()
test[['timestamp', 'congestion_index']].head()

## Load packaged inference artifacts

In [ ]:
pipeline = TrafficForecastingPipeline.from_artifacts(PROJECT_ROOT / 'models')

## Evaluate the held-out test period

In [ ]:
result = pipeline.backtest(test)
pd.DataFrame([result['model_metrics'], result['baseline_metrics']], index=['Stacked LSTM', 'Persistence baseline'])

## Actual vs predicted traffic

In [ ]:
predictions = result['predictions']
predictions.head()

In [ ]:
from src.visualization import actual_vs_predicted_figure
actual_vs_predicted_figure(predictions.head(300))

## Residual analysis

In [ ]:
from src.visualization import residual_figure
residual_figure(predictions.head(300))

## 24-hour recursive scenario forecast

In [ ]:
sample = pd.read_csv(PROJECT_ROOT / 'data' / 'sample_traffic_flow_data.csv')
future = pipeline.recursive_forecast(sample, horizon=24)
future

## Optional retraining

Install `requirements-dev.txt`, then run:

```bash
python train_model.py --epochs 20
```

Retraining is optional; the Streamlit app loads packaged artifacts and does not train during startup.